In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications.efficientnet import preprocess_input
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_real/*.jpg",shuffle=False)
train_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_fake/*.jpg",shuffle=False)
test_real = tf.data.Dataset.list_files( "/content/drive/MyDrive/face_detection_test/face_real/*.jpg",shuffle=False)
test_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",shuffle=False)


In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img
train_real = train_real.map(load_image)
train_fake = train_fake.map(load_image)
test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

In [ ]:
def add_label(image, label):
  return image , label

train_real = train_real.map(lambda x: add_label(x,0))
train_fake = train_fake.map(lambda x: add_label(x,1))
test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))

In [ ]:
train_dataset = train_real.concatenate(train_fake)
test_dataset = test_real.concatenate(test_fake)
train_dataset = train_dataset.shuffle(8000)
#test_dataset = test_dataset.shuffle(2000)
train_size = int(0.8 * 8000)   # 6400
val_size = 8000 - train_size   # 1600

validation_dataset = train_dataset.skip(train_size)
train_dataset = train_dataset.take(train_size)

In [ ]:
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label
train_dataset = train_dataset.map(preprocess)
validation_dataset = validation_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)


In [ ]:
BATCH_SIZE = 32
train_dataset = train_dataset.batch(BATCH_SIZE)
validation_dataset = validation_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
model = tf.keras.models.load_model("/content/drive/MyDrive/efficientnet_stage2.keras")
base_model = model.layers[1]
base_model.trainable = True
for layers in base_model.layers[:-60]:
  layers.trainable = False


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
              loss="binary_crossentropy",
              metrics=["accuracy"])


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/efficientnet_finetuned3.keras",
    monitor="val_accuracy",
    save_best_only=True
)

history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 3340s 273ms/step - accuracy: 0.5636 - loss: 0.6855 - val_accuracy: 0.5708 - val_loss: 0.6751
Epoch 2/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 56s 175ms/step - accuracy: 0.5669 - loss: 0.6834 - val_accuracy: 0.5996 - val_loss: 0.6684
Epoch 3/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 55s 165ms/step - accuracy: 0.5805 - loss: 0.6745 - val_accuracy: 0.5936 - val_loss: 0.6703
Epoch 4/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 54s 167ms/step - accuracy: 0.5833 - loss: 0.6721 - val_accuracy: 0.6123 - val_loss: 0.6586
Epoch 5/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 56s 168ms/step - accuracy: 0.5906 - loss: 0.6680 - val_accuracy: 0.6258 - val_loss: 0.6515
Epoch 6/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 82s 170ms/step - accuracy: 0.5995 - loss: 0.6619 - val_accuracy: 0.6559 - val_loss: 0.6452
Epoch 7/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 56s 170ms/step - accuracy: 0.6162 - loss: 0.6556 - val_accuracy: 0.6707 - val_loss: 0.6305
Epoch 8/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 57s 166ms/step - accuracy: 0.6306 - loss:

In [ ]:
model.evaluate(test_dataset)
print('accuracy:')

63/63 ━━━━━━━━━━━━━━━━━━━━ 849s 13s/step - accuracy: 0.5790 - loss: 0.6771
accuracy:
